# Tariff rates

In [1]:
import Pkg
Pkg.add("JuMP")
Pkg.add("HiGHS")

   Resolving package versions...
     Project No packages added to or removed from `C:\Users\huste\.julia\environments\v1.12\Project.toml`
    Manifest No packages added to or removed from `C:\Users\huste\.julia\environments\v1.12\Manifest.toml`
   Resolving package versions...
     Project No packages added to or removed from `C:\Users\huste\.julia\environments\v1.12\Project.toml`
    Manifest No packages added to or removed from `C:\Users\huste\.julia\environments\v1.12\Manifest.toml`


In [ ]:
T = 5
G = 3
demand = [15000 30000 25000 40000 27000]
avail = [12 10 5]
min_level = [850 1250 1500]
max_level = [2000 1750 4000]
cost_min = [1000 2600 3000]
cost_hour = [2 1.3 3]
start_up = [2000 1000 500]
duration = [6, 3, 6, 3, 6]

using JuMP, HiGHS
########## ---------- Models ---------- ##########
model = Model(HiGHS.Optimizer)
set_optimizer_attribute(model, "log_to_console", false)

########## ---------- Variables ---------- ##########

@variable(model, x[1:G, 1:T], Int) # No. off generatios
@variable(model, y[1:G, 1:T] >= 0) # Production total
@variable(model, z[1:G, 1:T] >= 0, Int) # Cahnge i


########## ---------- Objectives ---------- ##########
@objective(model, Min,
    sum(duration[t] * cost_min[g] * x[g,t] for g in 1:G, t in 1:T) +
    sum(duration[t] * cost_hour[g] * (y[g,t] - min_level[g] * x[g,t]) for g in 1:G, t in 1:T) +
    sum(start_up[g] * z[g,t] for g in 1:G, t in 1:T)
)


########## ---------- Constraint ---------- ##########
# Meet the demand
@constraint(model, [t in 1:T],
    sum(y[g,t] for g in 1:G) == demand[t]
)

# Be ready for 15% increasse
@constraint(model, [t in 1:T],
    sum(x[g,t] * max_level[g] for g in 1:G) >= demand[t]
)

# Bound y to x
@constraint(model, [g in 1:G, t in 1:T],
    x[g, t] * max_level[g] >= y[g,t]
)
@constraint(model, [g in 1:G, t in 1:T],
    x[g, t] * min_level[g] <= y[g,t]
)

# We cannot us more generators than we have available
@constraint(model, [g in 1:G, t in 1:T],
    x[g,t] <= avail[g]
)

# Get the no. of machines starting up
@constraint(model, [g in 1:G, t in 1:T],
    z[g, t] >= x[g,t] - (t > 1 ? x[g,t-1] : x[g,5])
)

########## ---------- Optimize ---------- ##########
optimize!(model)

println("Optimal solution:")
println("z = ", (objective_value(model)))

println("X : ")
for g in 1:G
    println(value.(x[g, :]))
end
println("Y : ")
for g in 1:G
    println(value.(y[g, :]))
end
println("Z : ")
for g in 1:G
    println(value.(z[g, :]))
end

Optimal solution:
z = 987790.0
X : 
[12.0, 12.0, 12.0, 12.0, 12.0]
[2.9999999999999987, 8.0, 8.0, 10.000000000000018, 9.0]
[-0.0, -0.0, -0.0, -5.921189464667502e-15, -0.0]
Y : 
[10200.0, 16000.0, 11000.0, 22499.999999999975, 11250.0]
[4800.0, 14000.0, 14000.0, 17500.000000000033, 15750.0]
[0.0, -0.0, -0.0, -8.881784197001252e-12, 0.0]
Z : 
[0.0, 0.0, 0.0, 0.0, 0.0]
[0.0, 5.000000000000002, 0.0, 2.0000000000000178, 0.0]
[0.0, 0.0, 0.0, -5.921189464667502e-15, 0.0]
